 ### AND OR SEARCH (Fully Observable Environment)

```python
function AND_OR_GRAPH_SEARCH(problem):
    return OR_SEARCH(problem.initial_state, problem, [])

function OR_SEARCH(state, problem, path):
    if state ∈ problem.goal_test:
        return []  // kế hoạch rỗng (đã đạt mục tiêu)
    if state ∈ path:
        return failure  // tránh lặp
    for each action in problem.actions(state):
        result_states = problem.results(state, action)
        plan = AND_SEARCH(result_states, problem, path + [state])
        if plan ≠ failure:
            return [action, plan]
    return failure

function AND_SEARCH(states, problem, path):
    plans = empty mapping  // lưu kế hoạch cho từng state
    for each s in states:
        plan_s = OR_SEARCH(s, problem, path)
        if plan_s == failure:
            return failure
        plans[s] = plan_s
    return plans

In [ ]:
import copy
class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action, gen: int):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.possible_moves= []
        self.gen = gen
        
    def get_tuple_floor_state(self):
        return tuple(tuple(row) for row in self.floor_state)
    
    def is_goal_state(self):
        return self.floor_state == GOAL_STATE

    def calculate_posible_moves(self):
        """
        This function return the possible move of the vacuum
        """
        if self.position[0] > 0: self.possible_moves.append("UP")
        if self.position[0] < ROW - 1: self.possible_moves.append("DOWN")
        if self.position[1] > 0: self.possible_moves.append("LEFT")
        if self.position[1] < COL - 1: self.possible_moves.append("RIGHT")

GOAL_STATE = [[0,0,0],
              [0,0,0],
              [0,0,0]]
ROW = 3
COL = 3

def apply_move(floor, vacuum_pos, move):
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos

def run_and_or_search(root: Node):
    return or_search(root, [])

def or_search(node: Node, path: list):
    if node.is_goal_state():
        return []
    
    state = (node.get_tuple_floor_state(), node.position)
    if state in path:
        return "FAILURE"
    
    node.calculate_posible_moves()
    for action in node.possible_moves:
        # results(state, action) trả về 2 kết quả khả thi do tỉ lệ 20% trượt chân:
        # 1. Di chuyển thành công sang ô mới (child)
        temp_floor, temp_vacuum_pos = apply_move(node.floor_state, node.position, action)
        child = Node(temp_floor, temp_vacuum_pos, node, action, node.gen + 1)
        
        # 2. Trượt chân đứng yên tại chỗ (node hiện tại)
        result_states = [child, node]
        
        plan = and_search(result_states, state, path + [state])
        if plan != "FAILURE":
            return [action, plan]
            
    return "FAILURE"

def and_search(nodes: list, parent_state, path: list):
    plans = {}
    for node in nodes:
        state = (node.get_tuple_floor_state(), node.position)
        if state == parent_state:
            # Gặp trạng thái tự lặp do trượt chân, ta gán kế hoạch "LOOP" để tránh lặp vô hạn
            plans[state] = "LOOP"
        else:
            plan = or_search(node, path)
            if plan == "FAILURE":
                return "FAILURE"
            plans[state] = plan
    return plans

def print_plan(plan, indent=0):
    if plan == []:
        print(" " * indent + "GOAL REACHED (Floor is completely clean!)")
        return

    if plan == "LOOP":
        print(" " * indent + "LOOP (Stayed in same state, retry action)")
        return
    
    action, contingencies = plan
    print(" " * indent + f"-> Action: {action}")
    for state, subplan in contingencies.items():
        floor, pos = state
        print(" " * indent + f"   If Outcome state is Floor={floor}, Position={pos}:")
        print_plan(subplan, indent + 6)

if __name__ == "__main__":
    import random
    
    # Initialize a random floor (3x3) where tiles have a 30% chance of being dirty
    # and a random starting position for the vacuum
    random.seed(42)  # Seed for reproducible test results
    
    initial_floor = []
    for _ in range(ROW):
        initial_floor.append([1 if random.random() < 0.3 else 0 for _ in range(COL)])
        
    start_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    
    # Ensure vacuum position starts clean (according to problem logic)
    initial_floor[start_pos[0]][start_pos[1]] = 0
    
    root_node = Node(initial_floor, start_pos, None, None, 0)
    
    print("=" * 60)
    print("VACUUM CLEANER ENVIRONMENT INITIAL STATE:")
    print(f"Floor State: {initial_floor}")
    print(f"Vacuum Position: {start_pos}")
    print("=" * 60)
    
    plan = run_and_or_search(root_node)
    
    if plan != "FAILURE":
        print("\nSUCCESS! FOUND A CONTINGENCY PLAN FOR THE 20% SLIP PROBABILITY:")
        print_plan(plan)
    else:
        print("\nFAILURE: Could not find a valid plan.")
    print("=" * 60)